# 07 — Полная Qwen/CrafText trajectory и визуализация

Тетрадка выполняет настоящий итоговый путь `CrafText state batch → MegaPrompts → batched Qwen/Tunix → strict decode/action-mask fallback → CrafText vmap(step) → per-env replay`. Она сохраняет versioned replay JSON и рисует один выбранный environment trace. Это host-side functional smoke; production PPO/DPO objectives подключаются позже через typed batch contracts.

До запуска явно подготовьте local Qwen snapshot и extras — тетрадка **никогда не скачивает веса**:

```bash
pyenv exec python -m uv sync --extra envs --extra prompts --extra tunix --extra examples
```


In [ ]:
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
import numpy as np

from tunix_craftext.rollouts.batched import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.env.prompts import MegaPromptRenderer
from tunix_craftext.artifacts.replay import save_replay
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.models.tunix_adapter import QwenTunixBackend


In [ ]:
ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / 'pyproject.toml').is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError('Run the notebook from inside the tunix-craftext repository.')

SNAPSHOT = ROOT / 'artifacts' / 'models' / 'qwen25-05b-instruct'
if not SNAPSHOT.is_dir():
    raise FileNotFoundError(
        f'Expected approved local Qwen snapshot at {SNAPSHOT}; no download is implicit.'
    )

config_path = ROOT / 'configs' / 'mvp' / 'qwen_craftext.yaml'
config = load_mvp_config(config_path)
runtime = build_craftext_runtime(config)
BATCH_SIZE = 2
HORIZON = 4  # Keep this small for an inspectable local smoke.
MAX_NEW_TOKENS = 8

def git_revision() -> str:
    try:
        return subprocess.check_output(
            ['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return 'unversioned'

print('actions:', runtime.actions.labels)
print('configured horizon:', HORIZON)


In [ ]:
# MegaPrompts receives structured EnvState internally; pixel observations are retained in replay.
renderer = MegaPromptRenderer(config.prompt.template)
backend = QwenTunixBackend(SNAPSHOT, cache_size=2048, seed=config.run.seed)
fallback_action_id = runtime.actions.index_of('NOOP')

rollout = collect_batched_text_rollout(
    runtime.adapter,
    renderer,
    backend,
    actions=runtime.actions,
    batch_size=BATCH_SIZE,
    horizon=HORIZON,
    seed=config.run.seed,
    goal='Stay alive, inspect the world and choose one currently valid action.',
    max_new_tokens=MAX_NEW_TOKENS,
    invalid_action='fallback',
    fallback_action_id=fallback_action_id,
)
replays = replays_from_batched_rollout(
    rollout,
    config_path=str(config_path.relative_to(ROOT)),
    commit=git_revision(),
    backend='tunix-single-device:Qwen',
)

output_dir = ROOT / 'artifacts' / 'trajectories' / 'qwen-craftext-full-notebook'
for env_index, replay in enumerate(replays):
    save_replay(output_dir / f'env-{env_index}.json', replay)
artifact = replays[0]
print(f'saved {len(replays)} env replays with {len(artifact.steps)} steps each to: {output_dir}')
print('backend:', artifact.backend)


In [ ]:
for step in artifact.steps:
    print(
        f'#{step.index}: action={step.action_label} reward={step.reward:.3f} '
        f'fallback={step.fallback_used} tokens={len(step.token_ids or ())}'
    )
    print('  completion:', repr(step.raw_completion))

assert (output_dir / 'env-0.json').is_file(), 'replay artifact must be persisted before visualization'


In [ ]:
steps = artifact.steps
indices = np.asarray([step.index for step in steps])
rewards = np.asarray([step.reward for step in steps], dtype=np.float32)
action_ids = np.asarray([step.action_id for step in steps], dtype=np.int32)
fallback = np.asarray([step.fallback_used for step in steps])

figure, (reward_axis, action_axis) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
reward_axis.plot(indices, rewards, marker='o', color='#4caf50', label='environment reward')
reward_axis.axhline(0.0, color='black', linewidth=0.8)
reward_axis.set_ylabel('reward')
reward_axis.legend()

colors = np.where(fallback, '#f59e0b', '#2563eb')
action_axis.bar(indices, action_ids, color=colors)
action_axis.set_xlabel('host decision step')
action_axis.set_ylabel('action id')
action_axis.set_yticks(np.arange(len(runtime.actions.labels)))
action_axis.set_yticklabels(runtime.actions.labels)
action_axis.set_title('Blue: model-decoded action; amber: explicit NOOP fallback')
figure.tight_layout()
plt.show()


In [ ]:
def image_for_display(observation: object) -> np.ndarray | None:
    """Turn a saved CrafText observation into a displayable HWC image."""
    image = np.asarray(observation)
    if image.ndim == 2:
        image = np.repeat(image[..., None], 3, axis=-1)
    if image.ndim != 3 or image.shape[-1] not in (1, 3, 4):
        return None
    image = image[..., :3].astype(np.float32)
    image -= image.min()
    maximum = image.max()
    return image / maximum if maximum else image

frames = [image_for_display(step.observation) for step in steps]
visible = [(step, frame) for step, frame in zip(steps, frames, strict=True) if frame is not None]
if not visible:
    raise RuntimeError('Replay does not contain image-like observations to render.')

figure, axes = plt.subplots(1, len(visible), figsize=(4 * len(visible), 4), squeeze=False)
for axis, (step, frame) in zip(axes.flat, visible, strict=True):
    axis.imshow(frame)
    marker = 'fallback' if step.fallback_used else 'model'
    axis.set_title(f'#{step.index} — {step.action_label}\n{marker}, r={step.reward:.2f}')
    axis.axis('off')
figure.suptitle('Observations retained in the saved replay')
figure.tight_layout()
plt.show()


## Что теперь можно проверить

- JSON в `artifacts/trajectories/qwen-craftext-full-notebook.json` содержит prompt, raw completion, action masks, observation, fallback flags и token provenance для каждого шага.
- Внешний интерактивный viewer можно открыть из корня репозитория: `PYTHONPATH=src .venv/bin/python scripts/visualize_trajectory.py --trajectory artifacts/trajectories/qwen-craftext-full-notebook.json`.
- Для learning path этот replay преобразуется через `text_trajectory_from_replay()`. Не используйте этот одиночный run как benchmark или как доказательство обучения.
